In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [43]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {"role": "user", "content": message.content if isinstance(message,Message) else message}
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {"role": "assistant", "content": message.content if isinstance(message,Message) else message}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[],tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message



def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [ ]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [6]:
from datetime import datetime
from anthropic.types import ToolParam


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)



get_current_datetime_schema =ToolParam(
    {
  "name": "get_current_datetime",
  "description": "Returns the current local date and time as a formatted string. Call this tool whenever you need the actual, real-time current date or time — for example to timestamp output, compute a relative date, determine the day of the week, or answer a question like 'what time is it right now?'. You have no built-in knowledge of the current date or time, so you must call this tool rather than guessing. The value is generated fresh at call time from the host system's local clock and time zone (not necessarily the user's time zone), so repeated calls can return different results. The output is a single string formatted according to the 'date_format' parameter using standard Python strftime directives (e.g. %Y = 4-digit year, %m = 2-digit month, %d = 2-digit day, %H = 24-hour, %M = minute, %S = second, %A = full weekday name). If 'date_format' is omitted, it defaults to '%Y-%m-%d %H:%M:%S', producing output like '2026-06-21 14:30:05'. Passing an empty string for 'date_format' is invalid and causes the tool to raise an error instead of returning a value.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A strftime-style format string controlling how the current date/time is rendered. Examples: '%Y-%m-%d' -> '2026-06-21', '%H:%M' -> '14:30', '%A, %B %d %Y' -> 'Sunday, June 21 2026'. Must be non-empty if provided.",
        "minLength": 1,
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "additionalProperties": False
  },
  "input_examples": [
    {},
    { "date_format": "%H:%M" },
    { "date_format": "%A, %B %d %Y" }
  ]
}
)

In [11]:
messages = []


messages.append(
    {

        "role":"user",
        "content":"What is the exact time , formatted as  HH:MM:SS?"

    }

)

response = client.messages.create(
    model = model,
    max_tokens=1000,
    messages=messages,
    tools = [get_current_datetime_schema],
)



messages.append({
    "role":"assistant",
    "content":response.content
})



print(messages)

[{'role': 'user', 'content': 'What is the exact time , formatted as  HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01Bz4szrVVzpPibiZVwmah5N', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


In [32]:
response.content[0].id

'toolu_01Bz4szrVVzpPibiZVwmah5N'

In [34]:
result = get_current_datetime(**response.content[0].input)

In [35]:
messages.append({
    "role":"user",
    "content":[
        {
            "type":"tool_result",
            "tool_use_id":response.content[0].id,
            "content":result,
            "is_error":False
        }

    ]


})

In [36]:
messages

[{'role': 'user',
  'content': 'What is the exact time , formatted as  HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01Bz4szrVVzpPibiZVwmah5N', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Bz4szrVVzpPibiZVwmah5N',
    'content': '12:38:44',
    'is_error': False}]}]

In [38]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_01XcbFpgryWBkhuq3Cp3wcDn', container=None, content=[TextBlock(citations=None, text='The exact time is **12:38:44** (HH:MM:SS format).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1208, output_tokens=23, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [42]:
messages = []


add_user_message(messages,"Whats the current time in HH:MM:SS format?")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

add_assistant_message(messages,response.content)


print(messages)

[{'role': 'user', 'content': 'Whats the current time in HH:MM:SS format?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01TXJbCPqXECteqCPXkcnFru', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


In [44]:
messages = []


add_user_message(messages,"Whats the current time in HH:MM:SS format?")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response

Message(id='msg_01FYSj3ty8K3KgnDFWQAKnyU', container=None, content=[ToolUseBlock(id='toolu_01RiYRHmrmM6jgMQ2uxxvLWP', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1127, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [45]:
import json


def run_tool(tool_name,tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)





def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []

    for tool_request in tool_requests:
       
        try:
            tool_output = run_tool(tool_request.name,tool_request.input)
            tool_result_block = {
                "type":"tool_result",
                "tool_use_id":tool_request.id,
                "content":json.dumps(tool_output),
                "is_error":False
            }
        except Exception as e:
            tool_result_block = {
                "type":"tool_result",
                "tool_use_id":tool_request.id,
                "content":f"Error:{e}",
                "is_error":True
            }

        tool_result_blocks.append(tool_result_block)


    return tool_result_blocks


    

In [46]:
def run_conversation(messages):
    while True:
        response = chat(messages,tools=[get_current_datetime_schema])

        add_assistant_message(messages,response)
        print(text_from_message(response))


        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages,tool_results)

    return messages


In [49]:
messages=[]

add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?")


run_conversation(messages)


The current time is:
- **HH:MM format:** 14:12
- **SS format:** 34 (seconds)


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01V9HcsyCvMb88veGULYCwAf', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01MPsuLnaFcXzTqudBV8A5Kb', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01V9HcsyCvMb88veGULYCwAf',
    'content': '"14:12"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01MPsuLnaFcXzTqudBV8A5Kb',
    'content': '"34"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format:** 14:12\n- **SS format:** 34 (seconds)', type='text')]}]